# Classificação de Desempenho Acadêmico com Redes Neurais (MLP)

Este projeto aplica uma **Rede Neural Perceptron Multicamadas (MLP)** utilizando **TensorFlow / Keras** para predizer o nível de desempenho escolar de estudantes a partir do dataset **xAPI-Edu-Data** (Kaggle).

Além da classificação multiclasse (`Alto`, `Médio` e `Baixo (Risco)`), o estudo realiza uma **análise comparativa por subgrupos de gênero** (Feminino vs. Masculino) para verificar variações na capacidade preditiva e padrões de erro.

## 1. Instalação das Dependências

In [ ]:
!pip install kagglehub[pandas-datasets] scikit-learn seaborn matplotlib tensorflow pandas numpy

## 2. Importação das Bibliotecas e Configuração

In [ ]:
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Fixando semente para reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Carregamento e Preparação dos Dados

In [ ]:
file_path = "xAPI-Edu-Data.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "aljarah/xAPI-Edu-Data",
    file_path,
)

# Mapeamento das classes do desempenho
traducao_classes = {'H': 'Alto', 'M': 'Medio', 'L': 'Baixo (Risco)'}
df['Class'] = df['Class'].map(traducao_classes)

print("Visualização das primeiras linhas:")
display(df.head())

## 4. Função de Treinamento e Avaliação da MLP

In [ ]:
def treinar_avaliar_mlp(df_subgrupo, nome_grupo):
    print(f"\n{'='*50}")
    print(f" TREINANDO MLP PARA O GRUPO: {nome_grupo.upper()}")
    print(f"{'='*50}")

    # Separar atributos preditores e o alvo
    X_bruto = df_subgrupo.drop(columns=['Class', 'gender'])
    y_bruto = df_subgrupo['Class'].values.reshape(-1, 1)

    # Converter variáveis categóricas em numéricas (One-Hot Encoding nas entradas)
    X = pd.get_dummies(X_bruto, drop_first=True).values.astype(np.float32)

    # Codificar a variável-alvo multiclasse (One-Hot Encoding nas saídas)
    encoder = OneHotEncoder(sparse_output=False)
    y = encoder.fit_transform(y_bruto)
    nomes_classes = encoder.categories_[0]

    # Divisão em Treino e Teste (80% treino / 20% teste)
    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # Normalização dos Dados (Z-score)
    padronizador = StandardScaler()
    X_treino = padronizador.fit_transform(X_treino)
    X_teste = padronizador.transform(X_teste)

    # Construção da MLP
    num_atributos = X_treino.shape[1]
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(num_atributos,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(y.shape[1], activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Treinamento da Rede Neural
    historico = model.fit(
        X_treino, y_treino,
        epochs=60, batch_size=16,
        validation_split=0.1, verbose=0
    )

    # Visualização do Histórico de Aprendizagem (Loss)
    plt.figure(figsize=(6, 4))
    plt.plot(historico.history['loss'], label='Treino')
    plt.plot(historico.history['val_loss'], label='Validação')
    plt.title(f'Histórico de Aprendizagem - Grupo {nome_grupo}')
    plt.ylabel('Erro (Loss)')
    plt.xlabel('Épocas')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Avaliação no Teste
    erro_teste, acuracia_teste = model.evaluate(X_teste, y_teste, verbose=0)
    print(f"\nAcurácia (Accuracy) no Teste: {acuracia_teste * 100:.2f}%\n")

    predicoes = model.predict(X_teste, verbose=0)
    pred_classes = np.argmax(predicoes, axis=1)
    y_teste_classes = np.argmax(y_teste, axis=1)

    # Relatório com Precisão, Sensibilidade (Recall) e F1-Score
    print(f"--- MÉTRICAS DE AVALIAÇÃO ({nome_grupo}) ---")
    print(classification_report(y_teste_classes, pred_classes, target_names=nomes_classes))

    # Matriz de Confusão
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(y_teste_classes, pred_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples' if nome_grupo == 'Feminino' else 'Blues',
                xticklabels=nomes_classes, yticklabels=nomes_classes)
    plt.title(f'Matriz de Confusão - {nome_grupo}')
    plt.xlabel('Predição da MLP')
    plt.ylabel('Classe Real')
    plt.tight_layout()
    plt.show()

    return acuracia_teste

## 5. Treinamento e Comparação dos Subgrupos

In [ ]:
# Separação dos subgrupos por gênero
df_feminino = df[df['gender'] == 'F'].copy()
df_masculino = df[df['gender'] == 'M'].copy()

# Executar o pipeline para os dois grupos
acc_fem = treinar_avaliar_mlp(df_feminino, "Feminino")
acc_masc = treinar_avaliar_mlp(df_masculino, "Masculino")

# Resumo comparativo
print("\n==========================================")
print("       RESUMO COMPARATIVO FINAL           ")
print("==========================================")
print(f"Acurácia MLP Grupo Feminino : {acc_fem * 100:.2f}%")
print(f"Acurácia MLP Grupo Masculino: {acc_masc * 100:.2f}%")